In [1]:
import pandas as pd

# Cargar el dataset desde la carpeta data relativa al notebook
df = pd.read_csv("data/housing.csv")

# Confirmar lectura
print("Dimensiones del dataset:", df.shape)
df.head()

Dimensiones del dataset: (20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Separar variables predictoras (X) y objetivo (y)
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

# Realizar la partición 80% Train / 20% Test (fijando semilla 42)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train_raw.shape}, Test set: {X_test_raw.shape}")

Train set: (16512, 9), Test set: (4128, 9)


In [3]:
# Guardar particiones crudas en fase-1/data/
X_train_raw.to_csv('data/X_train_raw.csv', index=False)
X_test_raw.to_csv('data/X_test_raw.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

In [4]:
# Entrenar DummyRegressor usando la media del set de entrenamiento
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train_raw, y_train)

# Predicciones sobre el set de prueba
y_pred_dummy = dummy.predict(X_test_raw)

# Calcular métricas de evaluación
mae_dummy = mean_absolute_error(y_test, y_pred_dummy)
rmse_dummy = np.sqrt(mean_squared_error(y_test, y_pred_dummy))
r2_dummy = r2_score(y_test, y_pred_dummy)

print(f"Baseline (Dummy) -> MAE: {mae_dummy:.2f}, RMSE: {rmse_dummy:.2f}, R2: {r2_dummy:.4f}")

# Registrar la métrica en el archivo resultados/metricas.csv
import csv
import os

# Crear la carpeta resultados si no existe
os.makedirs("resultados", exist_ok=True)

linea = [
    "Baseline_Mean",
    "Oscar",
    f"{mae_dummy:.2f}",
    f"{rmse_dummy:.2f}",
    f"{r2_dummy:.4f}",
]
with open("resultados/metricas.csv", mode="a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(linea)

print(
    "¡Métricas guardadas exitosamente en resultados/metricas.csv!"
)

Baseline (Dummy) -> MAE: 90606.85, RMSE: 114485.64, R2: -0.0002
¡Métricas guardadas exitosamente en resultados/metricas.csv!


## 4. Preparación de los datos — Samuel

Se parte del split crudo generado en la sección anterior. Todas las transformaciones que aprenden parámetros a partir de los datos (imputación, codificación) se ajustan (`fit`) únicamente sobre el conjunto de entrenamiento y luego se aplican (`transform`) al conjunto de prueba. El conjunto de prueba no interviene en ninguna decisión de preparación, para evitar fuga de información.

In [5]:
import pandas as pd
from sklearn.impute import SimpleImputer

X_train = pd.read_csv('data/X_train_raw.csv')
X_test = pd.read_csv('data/X_test_raw.csv')

print("Dimensiones -> train:", X_train.shape, "| test:", X_test.shape)
print("\nValores faltantes por variable (train):")
print(X_train.isnull().sum())
print(f"\ntotal_bedrooms: {X_train['total_bedrooms'].isnull().sum()} nulos "
      f"({X_train['total_bedrooms'].isnull().mean() * 100:.2f}% del train)")

Dimensiones -> train: (16512, 9) | test: (4128, 9)

Valores faltantes por variable (train):
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
ocean_proximity       0
dtype: int64

total_bedrooms: 0 nulos (0.00% del train)


### 4.1 Tratamiento de valores faltantes

`total_bedrooms` es la única variable con valores faltantes, en torno al 1% del conjunto.

Se imputa con la **mediana** y no con la media porque la distribución de la variable es fuertemente asimétrica a la derecha: existen distritos con un número muy alto de habitaciones que desplazan la media hacia arriba. La mediana es robusta frente a esos valores extremos y representa mejor al distrito típico.

Tampoco se eliminan las observaciones afectadas. Aunque el porcentaje es bajo, descartarlas implica perder distritos completos sin una razón que lo justifique, y la imputación por mediana introduce menos distorsión que esa pérdida de información.

El imputador se ajusta solo con los datos de entrenamiento y la mediana aprendida se aplica también al conjunto de prueba.

In [6]:
imputer = SimpleImputer(strategy='median')
imputer.fit(X_train[['total_bedrooms']])

X_train['total_bedrooms'] = imputer.transform(X_train[['total_bedrooms']])
X_test['total_bedrooms'] = imputer.transform(X_test[['total_bedrooms']])

print(f"Mediana aprendida del conjunto de entrenamiento: {imputer.statistics_[0]:.1f}")
print("Nulos restantes -> train:", X_train.isnull().sum().sum(),
      "| test:", X_test.isnull().sum().sum())

Mediana aprendida del conjunto de entrenamiento: 437.0
Nulos restantes -> train: 0 | test: 0


### 4.2 Codificación de la variable categórica

`ocean_proximity` es una variable categórica nominal: sus categorías (`<1H OCEAN`, `INLAND`, `NEAR OCEAN`, `NEAR BAY`, `ISLAND`) no tienen un orden natural. Asignarles números enteros introduciría una relación de orden y de magnitud que no existe en los datos, por lo que se aplica **one-hot encoding**.

Después de codificar se alinean las columnas del conjunto de prueba con las del conjunto de entrenamiento. Esto cubre el caso de que una categoría poco frecuente (por ejemplo `ISLAND`, con muy pocos registros) no aparezca en el conjunto de prueba y genere un número distinto de columnas.

In [7]:
X_train = pd.get_dummies(X_train, columns=['ocean_proximity'], dtype=int)
X_test = pd.get_dummies(X_test, columns=['ocean_proximity'], dtype=int)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

assert X_train.isnull().sum().sum() == 0, "Quedan nulos en train"
assert X_test.isnull().sum().sum() == 0, "Quedan nulos en test"
assert list(X_train.columns) == list(X_test.columns), "Columnas desalineadas"

print("Columnas finales:")
for c in X_train.columns:
    print(" -", c)
print("\nDimensiones -> train:", X_train.shape, "| test:", X_test.shape)

X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)
print("\nGuardados: data/X_train.csv y data/X_test.csv")

Columnas finales:
 - longitude
 - latitude
 - housing_median_age
 - total_rooms
 - total_bedrooms
 - population
 - households
 - median_income
 - ocean_proximity_<1H OCEAN
 - ocean_proximity_INLAND
 - ocean_proximity_ISLAND
 - ocean_proximity_NEAR BAY
 - ocean_proximity_NEAR OCEAN

Dimensiones -> train: (16512, 13) | test: (4128, 13)

Guardados: data/X_train.csv y data/X_test.csv


In [8]:
import pandas as pd

original = pd.read_csv('data/housing.csv')
print("housing.csv -> nulos en total_bedrooms:", original['total_bedrooms'].isnull().sum())

raw_train = pd.read_csv('data/X_train_raw.csv')
raw_test = pd.read_csv('data/X_test_raw.csv')
print("X_train_raw.csv -> nulos:", raw_train['total_bedrooms'].isnull().sum())
print("X_test_raw.csv  -> nulos:", raw_test['total_bedrooms'].isnull().sum())

housing.csv -> nulos en total_bedrooms: 207
X_train_raw.csv -> nulos: 0
X_test_raw.csv  -> nulos: 207


In [9]:
import pandas as pd
print("raw train:", pd.read_csv('data/X_train_raw.csv')['total_bedrooms'].isnull().sum())
print("raw test :", pd.read_csv('data/X_test_raw.csv')['total_bedrooms'].isnull().sum())

raw train: 0
raw test : 207


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/housing.csv')
print("housing.csv nulos:", df['total_bedrooms'].isnull().sum())

X = df.drop(columns=['median_house_value'])
y = df['median_house_value']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print("EN MEMORIA -> train:", X_tr['total_bedrooms'].isnull().sum(),
      "| test:", X_te['total_bedrooms'].isnull().sum())

X_tr.to_csv('data/_prueba.csv', index=False)
print("TRAS GUARDAR Y LEER ->",
      pd.read_csv('data/_prueba.csv')['total_bedrooms'].isnull().sum())

housing.csv nulos: 207


EN MEMORIA -> train: 0 | test: 207
TRAS GUARDAR Y LEER -> 0


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/housing.csv')
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

# Split por indices (mismo random_state, mismo tamano -> misma particion)
idx_train, idx_test = train_test_split(df.index, test_size=0.2, random_state=42)

X_tr2 = X.loc[idx_train]
X_te2 = X.loc[idx_test]

print("train:", X_tr2.shape, "| nulos:", X_tr2['total_bedrooms'].isnull().sum())
print("test :", X_te2.shape, "| nulos:", X_te2['total_bedrooms'].isnull().sum())

train: (16512, 9) | nulos: 0
test : (4128, 9) | nulos: 207


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/housing.csv')

# Posiciones exactas de las filas con nulo
nulos = df.index[df['total_bedrooms'].isnull()]
print("cantidad de nulos:", len(nulos))
print("primeras 10 posiciones:", list(nulos[:10]))
print("ultimas 10 posiciones:", list(nulos[-10:]))
print("posicion minima:", nulos.min(), "| maxima:", nulos.max())

idx_train, idx_test = train_test_split(df.index, test_size=0.2, random_state=42)
print("\nprimeros 10 indices de train:", list(idx_train[:10]))
print("primeros 10 indices de test :", list(idx_test[:10]))

print("\nnulos que caen en train:", len(set(nulos) & set(idx_train)))
print("nulos que caen en test :", len(set(nulos) & set(idx_test)))

cantidad de nulos: 207
primeras 10 posiciones: [290, 341, 538, 563, 696, 738, 1097, 1350, 1456, 1493]
ultimas 10 posiciones: [19932, 19959, 20046, 20069, 20125, 20267, 20268, 20372, 20460, 20484]
posicion minima: 290 | maxima: 20484

primeros 10 indices de train: [14196, 8267, 17445, 14265, 2271, 17848, 6252, 9389, 6113, 6061]
primeros 10 indices de test : [20046, 3024, 15663, 20484, 9814, 13311, 7113, 7668, 18246, 5723]

nulos que caen en train: 0
nulos que caen en test : 207


In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/housing.csv')
nulos = set(df.index[df['total_bedrooms'].isnull()])

idx_train, idx_test = train_test_split(df.index, test_size=0.2, random_state=42)
tr, te = set(idx_train), set(idx_test)

print("len train:", len(idx_train), "| unicos:", len(tr))
print("len test :", len(idx_test), "| unicos:", len(te))
print("solapamiento train/test:", len(tr & te))
print("cobertura total:", len(tr | te), "de", len(df))

# Comparacion: particion manual con numpy, sin sklearn
rng = np.random.RandomState(42)
perm = rng.permutation(len(df))
corte = int(len(df) * 0.8)
tr2, te2 = set(perm[:corte]), set(perm[corte:])
print("\nnumpy -> nulos en train:", len(nulos & tr2), "| en test:", len(nulos & te2))

len train: 16512 | unicos: 16512
len test : 4128 | unicos: 4128
solapamiento train/test: 0
cobertura total: 20640 de 20640

numpy -> nulos en train: 207 | en test: 0


In [14]:
with open('data/housing.csv', 'r', encoding='utf-8') as f:
    lineas = f.readlines()

encabezado = lineas[0].strip().split(',')
print("columnas:", encabezado)
pos = encabezado.index('total_bedrooms')
print("total_bedrooms es la columna numero:", pos)

filas_vacias = []
for i, linea in enumerate(lineas[1:], start=0):
    campos = linea.rstrip('\n').split(',')
    if len(campos) > pos and campos[pos].strip() == '':
        filas_vacias.append(i)

print("\nfilas con total_bedrooms vacio:", len(filas_vacias))
print("primeras 10:", filas_vacias[:10])
print("ultimas 10:", filas_vacias[-10:])

columnas: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']
total_bedrooms es la columna numero: 4

filas con total_bedrooms vacio: 207
primeras 10: [290, 341, 538, 563, 696, 738, 1097, 1350, 1456, 1493]
ultimas 10: [19932, 19959, 20046, 20069, 20125, 20267, 20268, 20372, 20460, 20484]


In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/housing.csv')
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

nulos_reales = set(int(i) for i in filas_vacias)
tr = set(int(i) for i in X_train_raw.index)
te = set(int(i) for i in X_test_raw.index)

print("VERDAD (posiciones del archivo)")
print("  nulos en train:", len(nulos_reales & tr))
print("  nulos en test :", len(nulos_reales & te))
print("CONTEO DE PANDAS")
print("  train:", X_train_raw['total_bedrooms'].isnull().sum())
print("  test :", X_test_raw['total_bedrooms'].isnull().sum())

VERDAD (posiciones del archivo)
  nulos en train: 0
  nulos en test : 207
CONTEO DE PANDAS
  train: 0
  test : 207


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/housing.csv')
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

for semilla in [0, 1, 7, 42, 123]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=semilla)
    print(f"seed {semilla:>4} -> train: {Xtr['total_bedrooms'].isnull().sum():>3} "
          f"| test: {Xte['total_bedrooms'].isnull().sum():>3}")

seed    0 -> train: 158 | test:  49
seed    1 -> train: 163 | test:  44
seed    7 -> train: 156 | test:  51
seed   42 -> train:   0 | test: 207
seed  123 -> train: 172 | test:  35


In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/housing.csv')
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

# Los indices de test que tienen nulo, segun el archivo crudo
nulos = set(df.index[df['total_bedrooms'].isnull()])
print("nulos totales:", len(nulos))
print("en test:", len(nulos & set(Xte.index)))
print("\nrevision manual de 5 filas nulas que cayeron en test:")
for i in list(nulos & set(Xte.index))[:5]:
    print(f"  fila {i}: valor en df = {df.loc[i, 'total_bedrooms']!r}, "
          f"valor en Xte = {Xte.loc[i, 'total_bedrooms']!r}")

print("\nseed 42 repetido 3 veces:")
for _ in range(3):
    a, b, c, d = train_test_split(X, y, test_size=0.2, random_state=42)
    print("  train:", a['total_bedrooms'].isnull().sum(), "| test:", b['total_bedrooms'].isnull().sum())

nulos totales: 207
en test: 207

revision manual de 5 filas nulas que cayeron en test:
  fila 7168: valor en df = np.float64(nan), valor en Xte = np.float64(nan)
  fila 17923: valor en df = np.float64(nan), valor en Xte = np.float64(nan)
  fila 20484: valor en df = np.float64(nan), valor en Xte = np.float64(nan)
  fila 17928: valor en df = np.float64(nan), valor en Xte = np.float64(nan)
  fila 10761: valor en df = np.float64(nan), valor en Xte = np.float64(nan)

seed 42 repetido 3 veces:
  train: 0 | test: 207
  train: 0 | test: 207
  train: 0 | test: 207


In [18]:
import sys, pandas, numpy, sklearn
print("python:", sys.version.split()[0])
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("sklearn:", sklearn.__version__)

python: 3.13.7
pandas: 3.0.5
numpy: 2.5.3
sklearn: 1.9.1
